### Environment Setup and Raw Data Ingestion
This cell imports required libraries (`pandas`) and loads all five raw CSV files into individual DataFrames without making any modifications.
Expected Output: Five DataFrames initialized and raw row counts printed for verification.

In [2]:
import pandas as pd

# Load raw datasets
df_appointments = pd.read_csv('data/appointments_raw.csv')
df_doctors = pd.read_csv('data/doctors.csv')
df_clinics = pd.read_csv('data/clinics.csv')
df_patients = pd.read_csv('data/patients.csv')
df_updates = pd.read_csv('data/patient_updates.csv')

print(f"appointments_raw rows: {len(df_appointments)}")
print(f"doctors rows: {len(df_doctors)}")
print(f"clinics rows: {len(df_clinics)}")
print(f"patients rows: {len(df_patients)}")
print(f"patient_updates rows: {len(df_updates)}")

appointments_raw rows: 1524
doctors rows: 21
clinics rows: 5
patients rows: 300
patient_updates rows: 3


### Data Quality Profiling
This cell inspects missing values, exact duplicates, grain-level duplicates, and invalid numeric/categorical values across all raw datasets using built-in pandas methods (`info()`, `isnull()`, `duplicated()`, `value_counts()`).
Expected Output: Displayed summary of identified data quality anomalies and row counts.

In [3]:
# 1. Missing Values Count
print("--- Missing Values per File ---")
print("Appointments:\n", df_appointments.isnull().sum(), "\n")
print("Doctors:\n", df_doctors.isnull().sum(), "\n")

# 2. Duplicate Detection
full_exact_dups = df_appointments.duplicated().sum()
grain_dups = df_appointments.duplicated(subset=['patient_id', 'doctor_id', 'clinic_id', 'appointment_date']).sum()
print(f"Exact full-row duplicates: {full_exact_dups}")
print(f"Grain-level duplicates (same patient, doctor, clinic, date): {grain_dups}\n")

# 3. Specialty Formatting Check
print("--- Distinct Specialties in doctors.csv ---")
print(df_doctors['specialty'].value_counts(), "\n")

# 4. Measure Validation (Wait, Consultation, Fee)
print("--- Appointments Measures Summary ---")
print(df_appointments[['wait_minutes', 'consultation_minutes', 'fee']].describe())

--- Missing Values per File ---
Appointments:
 appointment_id           0
patient_id               0
doctor_id                0
clinic_id                0
appointment_date         0
wait_minutes             0
consultation_minutes     0
fee                     15
dtype: int64 

Doctors:
 doctor_id      0
doctor_name    0
specialty      0
dtype: int64 

Exact full-row duplicates: 0
Grain-level duplicates (same patient, doctor, clinic, date): 24

--- Distinct Specialties in doctors.csv ---
specialty
General Medicine     4
Paediatrics          3
Cardiology           2
Orthopaedics         2
Neurology            2
Dermatology          2
dermatology          1
CARDIOLOGY           1
 Orthopaedics        1
cardiology           1
Neurology            1
general medicine     1
Name: count, dtype: int64 

--- Appointments Measures Summary ---
       wait_minutes  consultation_minutes          fee
count   1524.000000           1524.000000  1509.000000
mean      33.403543             21.660761  371

### Remove Repeated Bookings
This cell removes duplicate records based on the fact table grain (one patient seeing one doctor at one clinic on one date). 
Expected Output: The row count will decrease by 24.

**Explanation for Report:** 
Using `duplicated()` with no columns listed checks if the *entire row* is exactly identical. It found 0 duplicates because the erroneous rows have the same grain columns but different measure values (e.g., a different wait time or fee recorded for the exact same appointment). By specifying the subset `['patient_id', 'doctor_id', 'clinic_id', 'appointment_date']`, we correctly identify and remove the 24 grain-level violations.

In [4]:
print(f"Row count BEFORE deduplication: {len(df_appointments)}")

# Drop duplicates keeping the first occurrence based on the grain
df_appointments = df_appointments.drop_duplicates(
    subset=['patient_id', 'doctor_id', 'clinic_id', 'appointment_date'], 
    keep='first'
)

print(f"Row count AFTER deduplication: {len(df_appointments)}")

Row count BEFORE deduplication: 1524
Row count AFTER deduplication: 1500


### 3.3 Convert Date Formats
This cell standardizes the `appointment_date` column into a unified datetime format.
Expected Output: The column type will change, and a check will show 0 missing/failed conversions.

In [5]:
# Convert to standard datetime format (format='mixed' handles various date layouts)
df_appointments['appointment_date'] = pd.to_datetime(df_appointments['appointment_date'], format='mixed')

# Prove that no dates failed to convert (if they failed, they would become 'NaT' / Null)
failed_dates = df_appointments['appointment_date'].isnull().sum()
print(f"Number of dates that failed to convert: {failed_dates}")

Number of dates that failed to convert: 0


### 3.4 Standardize Specialty Text
This cell cleans the `specialty` text in the doctors dataset to a consistent Title Case format and removes trailing spaces.
Expected Output: The distinct count of specialties will be printed before and after, showing consolidation of messy categories.

In [6]:
print("--- Distinct specialties BEFORE cleaning ---")
print(df_doctors['specialty'].value_counts())

# Standardize to Title Case and strip extra whitespace
df_doctors['specialty'] = df_doctors['specialty'].str.title().str.strip()

print("\n--- Distinct specialties AFTER cleaning ---")
print(df_doctors['specialty'].value_counts())

--- Distinct specialties BEFORE cleaning ---
specialty
General Medicine     4
Paediatrics          3
Cardiology           2
Orthopaedics         2
Neurology            2
Dermatology          2
dermatology          1
CARDIOLOGY           1
 Orthopaedics        1
cardiology           1
Neurology            1
general medicine     1
Name: count, dtype: int64

--- Distinct specialties AFTER cleaning ---
specialty
General Medicine    5
Cardiology          4
Dermatology         3
Paediatrics         3
Orthopaedics        3
Neurology           3
Name: count, dtype: int64


### 3.5 Cleanse Measures
**Rules Applied:**
1. `wait_minutes` cannot be less than 0.
2. `consultation_minutes` must be greater than 0.
3. `fee` must not be null (NaN) and cannot be negative.
I will apply these rules by dropping the invalid rows to ensure data integrity.
Expected Output: Print the number of rows changed/dropped by these rules.

In [7]:
print(f"Row count BEFORE cleansing measures: {len(df_appointments)}")

# 1. Identify violations
invalid_wait = df_appointments[df_appointments['wait_minutes'] < 0].shape[0]
invalid_cons = df_appointments[df_appointments['consultation_minutes'] <= 0].shape[0]
missing_fee = df_appointments['fee'].isnull().sum()
invalid_fee = df_appointments[df_appointments['fee'] < 0].shape[0]

print(f"Violations found -> Negative wait: {invalid_wait}, Invalid consultation: {invalid_cons}, Missing fee: {missing_fee}, Negative fee: {invalid_fee}")

# 2. Apply the rules (keep only valid data)
df_appointments = df_appointments[
    (df_appointments['wait_minutes'] >= 0) & 
    (df_appointments['consultation_minutes'] > 0) & 
    (df_appointments['fee'].notnull()) & 
    (df_appointments['fee'] >= 0)
]

print(f"Row count AFTER cleansing measures: {len(df_appointments)}")

Row count BEFORE cleansing measures: 1500
Violations found -> Negative wait: 8, Invalid consultation: 0, Missing fee: 15, Negative fee: 0
Row count AFTER cleansing measures: 1477


### 3.6 Integrate with Doctors
This cell handles the join between appointments and doctors.
**Cause of row change:** If we join directly, the row count increases because there are duplicate `doctor_id` values in `doctors.csv`.
**Fix:** Drop duplicates in the `doctors` dataframe based on `doctor_id` before merging.

In [8]:
# Check and remove duplicates in doctors
print(f"Duplicate doctor IDs in doctors.csv: {df_doctors.duplicated(subset=['doctor_id']).sum()}")
df_doctors_cleaned = df_doctors.drop_duplicates(subset=['doctor_id'], keep='first')

# Merge appointments with cleaned doctors
df_merged = pd.merge(df_appointments, df_doctors_cleaned, on='doctor_id', how='left')

print(f"Final integrated appointments row count: {len(df_merged)}")

Duplicate doctor IDs in doctors.csv: 1
Final integrated appointments row count: 1477


### 4.1 Surrogate Keys and CSV Generation
This cell generates an auto-incrementing surrogate key (SK) for each dimension. It then maps these SKs back to the fact table, drops the original descriptive columns, and exports one clean CSV per table.

In [10]:
# 1. Generate Surrogate Keys for Dimensions
df_patients['patient_sk'] = range(1, len(df_patients) + 1)
df_doctors_cleaned['doctor_sk'] = range(1, len(df_doctors_cleaned) + 1)
df_clinics['clinic_sk'] = range(1, len(df_clinics) + 1)

# 2. Map SKs to the Fact Table
fact_df = pd.merge(df_merged, df_patients[['patient_id', 'patient_sk']], on='patient_id', how='left')
fact_df = pd.merge(fact_df, df_doctors_cleaned[['doctor_id', 'doctor_sk']], on='doctor_id', how='left')
fact_df = pd.merge(fact_df, df_clinics[['clinic_id', 'clinic_sk']], on='clinic_id', how='left')
fact_df['date_sk'] = fact_df['appointment_date'].dt.strftime('%Y%m%d').astype(int)

# 3. Isolate ONLY the columns required for the Fact Table
fact_final = fact_df[['patient_sk', 'doctor_sk', 'clinic_sk', 'date_sk', 
                      'wait_minutes', 'consultation_minutes', 'fee']]

# Rename 'fee' to 'fee_lkr' to match your PostgreSQL DDL exactly
fact_final = fact_final.rename(columns={'fee': 'fee_lkr'})

# 4. Export to CSV (Saved in the 'data' folder)
df_patients.to_csv('data/dim_patient_clean.csv', index=False)
df_doctors_cleaned.to_csv('data/dim_doctor_clean.csv', index=False)
df_clinics.to_csv('data/dim_clinic_clean.csv', index=False)
fact_final.to_csv('data/fact_appointment_clean.csv', index=False)

print("All 4 files successfully exported to CSV!")

All 4 files successfully exported to CSV!


In [11]:
import os
print("Exact path:", os.path.abspath('data/dim_patient_clean.csv'))

Exact path: c:\Users\Victus\Documents\DW\DW-CW-252\data\dim_patient_clean.csv


In [13]:
# Reorder columns so surrogate key is the FIRST column
df_patients = df_patients[['patient_sk', 'patient_id', 'gender', 'date_of_birth', 'district']]
df_doctors_cleaned = df_doctors_cleaned[['doctor_sk', 'doctor_id', 'doctor_name', 'specialty']]
df_clinics = df_clinics[['clinic_sk', 'clinic_id', 'clinic_name', 'city', 'province']]

# Re-export to CSV
df_patients.to_csv('data/dim_patient_clean.csv', index=False)
df_doctors_cleaned.to_csv('data/dim_doctor_clean.csv', index=False)
df_clinics.to_csv('data/dim_clinic_clean.csv', index=False)
fact_final.to_csv('data/fact_appointment_clean.csv', index=False)

print("CSVs re-exported with correct column order!")

CSVs re-exported with correct column order!
